# Generating PPT entangled states via SDP

Implementation of the construction from *"A simple class of bound entangled
states based on the properties of the antisymmetric subspace"* (Sindici & Piani).

**Idea.** For a random antisymmetric state ρ_A, solve an SDP for the largest
probability p of producing ρ_A by projecting a PPT state onto the antisymmetric
subspace. If that optimum p < 1/2, the optimal PPT state is **entangled** — a
bound entangled state we can feed into the PPT² test.

Projectors, `is_ppt`, and the detection criteria (`detect_trace`,
`detect_ampliation`, `detect_dps`) all come from the `ppt2` library.

In [ ]:
include("common.jl")
using Statistics

## Random antisymmetric states

Project a random density matrix onto the antisymmetric subspace P_A
(`antisymmetric_projector` from `ppt2`) and renormalise.

In [ ]:
function rand_state(d; rng=Random.GLOBAL_RNG)
    A = randn(rng, d, d)
    ρ = A * A'
    return ρ / tr(ρ)
end

function rand_antisymmetric_state(d; rng=Random.GLOBAL_RNG)
    ρ = rand_state(d^2; rng=rng)
    P_A = antisymmetric_projector(d)
    ρ_A = P_A * ρ * P_A
    return ρ_A / tr(ρ_A)
end

## SDP formulation

Maximise the antisymmetric-projection probability

$$\max_\sigma\ \mathrm{Tr}(P_A\,\sigma)$$

subject to $P_A\,\sigma\,P_A = \mathrm{Tr}(P_A\sigma)\,\rho_A$, $\sigma \ge 0$,
$\mathrm{Tr}(\sigma) = 1$, and $\sigma^{\Gamma} \ge 0$ (PPT). When the optimum
$p^{PPT}(\rho_A) < 1/2$, the optimal $\sigma$ is PPT **and** entangled.

In [ ]:
function solve_ppt_sdp(ρ_A, d; verbose=false)
    P_A = antisymmetric_projector(d)
    model = Model(Mosek.Optimizer)
    verbose || set_silent(model)

    @variable(model, σ[1:d^2, 1:d^2], PSD)
    @variable(model, p >= 0)
    @objective(model, Max, p)
    @constraint(model, tr(σ) == 1)
    @constraint(model, P_A * σ * P_A .== p .* ρ_A)
    @constraint(model, partial_transpose(σ, 1) in PSDCone())

    optimize!(model)
    return value(p), value.(σ)
end

## Theoretical bounds

For any antisymmetric $\rho_A$,

$$\frac{2}{d(d+1)+2}\ \le\ p^{PPT}(\rho_A)\ \le\ \frac{1}{2}.$$

The lower bound is attained by the family $\sigma(p) = p\,\rho_A \oplus (1-p)\,P_S/d_S$.

In [ ]:
lower_bound_ppt(d) = 2 / (d * (d + 1) + 2)
upper_bound_ppt(d) = 0.5

function construct_ppt_family(ρ_A, d, p)
    P_S = ppt2.symmetric_projector(d)   # qualified: Ket also exports this name
    d_S = d * (d + 1) ÷ 2
    σ = p .* ρ_A .+ (1 - p) .* P_S ./ d_S
    return σ / tr(σ)
end

## Generate a PPT entangled state

Draw antisymmetric states until the SDP returns a PPT optimum with p < 1/2.

In [ ]:
function generate_ppt_entangled_state(d; rng=Random.GLOBAL_RNG, max_attempts=100)
    for _ in 1:max_attempts
        ρ_A = rand_antisymmetric_state(d; rng=rng)
        p_opt, σ = solve_ppt_sdp(ρ_A, d)
        if is_ppt(σ, d, d) && p_opt < 0.5 - 1e-6
            return σ, ρ_A, p_opt    # PPT and entangled (Sindici–Piani, Lemma 1)
        end
    end
    return nothing, nothing, nothing
end

In [ ]:
d = 4
σ, ρ_A, p_opt = generate_ppt_entangled_state(d; rng=Xoshiro(42), max_attempts=2000)
(p_opt = p_opt, ppt = is_ppt(σ, d, d), lower_bound = lower_bound_ppt(d))

## Feed into the PPT² test

Compose the generated bound entangled state with itself and report all three
detection criteria (they should stay on the separable side).

In [ ]:
forms = load_batches("../pncp_4x4.jld2")
τ = Hermitian(ampliation(σ, σ, 4, 4))
(trace      = detect_trace(τ, forms).value,
 ampliation = detect_ampliation(τ, forms, 4, 4).value,
 dps        = detect_dps(τ, 4, 4).value)